In [ ]:
# ==== attach & run (safe) ====
from whisper import load_model
from whisper.tokenizer import get_tokenizer
from whisper.decoding import BeamSearchDecoder

from src.beam_hook import make_beam_update_hook, beam_log

# 1) モデル・トークナイザ
model = load_model("large-v3-turbo")
model.eval()
tokenizer = get_tokenizer(multilingual=True, task="transcribe")

# 2) ドメイン語彙
domain_terms = [
    "傷病者", "周囲の安全", "感染防御", "胸骨圧迫", "AED", "呼吸", "反応なし",
    "気道確保", "意識レベル", "バイタル", "心停止", "除細動", "脈拍", "瞳孔"
]

# 3) オリジナル update を退避 → フックを作成 → 置換
if not getattr(BeamSearchDecoder, "_hook_installed", False):
    _original_update = BeamSearchDecoder.update         # ★退避
    beam_update = make_beam_update_hook(tokenizer, domain_terms, _original_update)
    BeamSearchDecoder.update = beam_update              # ★置換
    BeamSearchDecoder._hook_installed = True            # 二重装着防止

# 4) 推論
WAV = "/root/MedWhisper/202502/右後ろ_1回目.wav"
result = model.transcribe(
    WAV,
    language="ja",
    task="transcribe",
    beam_size=3,
    temperature=0.0,
    condition_on_previous_text=False,
    #initial_prompt="傷病者 周囲の安全 感染防御 胸骨圧迫 AED"
)

print("=== 出力 ===")
print(result["text"])

# 5) ログ保存
import pandas as pd, os
os.makedirs("./_beamlog", exist_ok=True)
df = pd.DataFrame(beam_log)
csv_path = "./_beamlog/beam_steps.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")
print("saved:", csv_path)


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:309: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


=== 出力 ===
傷病者 周囲の安全 感染防御 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 胸骨圧迫 ご視聴ください。やっと、呼吸の6人。やけなし、呼吸なし。ここで発動します。3、4、5、6、7、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8、8パッドを胸に装着してくださいランプが点滅しているソケットにパッドのコレクターを接続してくださいパッドを装着してくださいコレクターを接続してください心電図を解析中です体に触れないでくださいショックが必要です充電中です体から離れてください入れてくださいショップを実行しますオレンジボタンを押してくださいショップが完了しました一時中断中ですただし胸骨圧迫と実行をお見せしますこの人が3分前に倒れているのを発表しました胸骨圧迫と電気ショップを行いましたこの人の目元は分かりませんがそこにこの人のバッグがあります来てきます来てます
saved: ./_beamlog/beam_steps.csv
